In [ ]:
# ============================================================
# Transformer Seq2Seq — Nepali Abstractive Summarization
# ============================================================
# Dependencies (install once):
#   pip install torch pandas numpy scikit-learn sentencepiece \
#               sacrebleu bert-score tqdm
#
# Environment:
#   NEPALI_DATA_DIR   folder containing small.csv, medium.csv, large.csv
#                     (default: ./data/)
#   NEPALI_WORK_DIR   output folder (default: ./transformer_workspace/)
#   NEPALI_TOKENIZER  optional path to an existing SentencePiece .model
#
# Reproducibility: BPE tokenizer is trained on the train split only.
# Metrics: SacreBLEU (flores200, 13a), ROUGE-1/2/L (manual F1),
#          BERTScore (ne), Jaccard (word-unigram, char-bigram).
# ============================================================

import os
import math
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score_fn
from tqdm.auto import tqdm


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
CONFIG = {
    "data_dir":  Path(os.environ.get("NEPALI_DATA_DIR", "./data")),
    "work_dir":  Path(os.environ.get("NEPALI_WORK_DIR", "./transformer_workspace")),
    "tokenizer": os.environ.get("NEPALI_TOKENIZER"),

    "test_size":      0.2,
    "seed":           42,
    "max_src_len":    400,
    "max_tgt_len":    64,
    "spm_vocab_size": 32000,

    "d_model":  256,
    "nhead":    8,
    "num_enc":  4,
    "num_dec":  4,
    "dim_ff":   1024,
    "dropout":  0.2,
    "max_pos":  512,

    "batch_size":   32,
    "epochs":       30,
    "patience":     8,
    "lr_peak":      3e-4,
    "warmup":       2000,
    "grad_clip":    1.0,
    "label_smooth": 0.1,

    "beam_size":  4,
    "eval_batch": 16,
}


# ------------------------------------------------------------
# Reproducibility and runtime
# ------------------------------------------------------------
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("CUDA GPU required.")

WORK_DIR     = CONFIG["work_dir"]
WORK_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH    = WORK_DIR / "transformer_best.pt"
TEST_RAW_CSV = WORK_DIR / "transformer_test_raw.csv"
PRED_CSV     = WORK_DIR / "transformer_test_predictions.csv"
SCORES_CSV   = WORK_DIR / "transformer_final_scores.csv"


# ------------------------------------------------------------
# Data: load, clean, split
# ------------------------------------------------------------
def load_and_split(cfg):
    """Read the three CSV shards, clean, and return 80/10/10 splits."""
    data_dir = cfg["data_dir"]
    required = ("small.csv", "medium.csv", "large.csv")
    missing = [n for n in required if not (data_dir / n).is_file()]
    if missing:
        raise FileNotFoundError(
            f"Missing {missing} in {data_dir.resolve()}. "
            f"Set NEPALI_DATA_DIR or place the CSVs in ./data/."
        )

    df = pd.concat([pd.read_csv(data_dir / n) for n in required], ignore_index=True)
    df = df.dropna(subset=["content", "summary"]).copy()
    df["content"] = df["content"].astype(str).str.strip()
    df["summary"] = df["summary"].astype(str).str.strip()
    df = df[(df["content"].str.len() > 10) & (df["summary"].str.len() > 5)]
    df = df.drop_duplicates(subset=["content"], keep="first").reset_index(drop=True)
    print(f"Cleaned rows: {len(df)}")

    train_df, temp_df = train_test_split(
        df, test_size=cfg["test_size"], random_state=cfg["seed"], shuffle=True
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.5, random_state=cfg["seed"], shuffle=True
    )
    print(f"Train/Val/Test: {len(train_df)}/{len(val_df)}/{len(test_df)}")
    return train_df, val_df, test_df


train_df, val_df, test_df = load_and_split(CONFIG)
test_df[["content", "summary"]].to_csv(TEST_RAW_CSV, index=False, encoding="utf-8")


# ------------------------------------------------------------
# SentencePiece BPE tokenizer
# ------------------------------------------------------------
def build_or_load_tokenizer(cfg, train_frame):
    """Load an existing tokenizer or train one on the train split only."""
    candidates = []
    if cfg["tokenizer"]:
        candidates.append(Path(cfg["tokenizer"]))
    candidates.append(WORK_DIR / "nepali_sp.model")
    for path in candidates:
        if path.is_file():
            print(f"Tokenizer: {path}")
            return spm.SentencePieceProcessor(model_file=str(path))

    print(f"Training SentencePiece BPE (vocab={cfg['spm_vocab_size']}) on train split...")
    corpus_path = WORK_DIR / "corpus.txt"
    with corpus_path.open("w", encoding="utf-8") as f:
        for t in train_frame["content"].astype(str):
            f.write(t.replace("\n", " ") + "\n")
        for t in train_frame["summary"].astype(str):
            f.write(t.replace("\n", " ") + "\n")

    model_prefix = WORK_DIR / "nepali_sp"
    spm.SentencePieceTrainer.train(
        input=str(corpus_path),
        model_prefix=str(model_prefix),
        vocab_size=cfg["spm_vocab_size"],
        model_type="bpe",
        character_coverage=1.0,
        max_sentence_length=100000,
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        split_by_whitespace=True,
        split_by_unicode_script=True,
        num_threads=os.cpu_count(),
        shuffle_input_sentence=True,
    )
    corpus_path.unlink()
    print(f"Tokenizer: {model_prefix}.model")
    return spm.SentencePieceProcessor(model_file=f"{model_prefix}.model")


sp = build_or_load_tokenizer(CONFIG, train_df)
PAD_ID, BOS_ID, EOS_ID = sp.pad_id(), sp.bos_id(), sp.eos_id()
VOCAB_SIZE = sp.get_piece_size()
print(f"Vocab: {VOCAB_SIZE} | PAD={PAD_ID} BOS={BOS_ID} EOS={EOS_ID}")


# ------------------------------------------------------------
# Encode text to token ids
# ------------------------------------------------------------
def encode(df, sp_proc, cfg):
    df = df.copy()
    df["src_ids"] = df["content"].astype(str).apply(
        lambda x: sp_proc.encode(x, out_type=int)[:cfg["max_src_len"]]
    )
    df["tgt_ids"] = df["summary"].astype(str).apply(
        lambda x: [BOS_ID]
        + sp_proc.encode(x, out_type=int)[:cfg["max_tgt_len"] - 2]
        + [EOS_ID]
    )
    return df


train_df, val_df, test_df = (encode(d, sp, CONFIG) for d in (train_df, val_df, test_df))


# ------------------------------------------------------------
# Dataset and collator
# ------------------------------------------------------------
class SummarizationDataset(Dataset):
    def __init__(self, df):
        self.src = df["src_ids"].tolist()
        self.tgt = df["tgt_ids"].tolist()

    def __len__(self):
        return len(self.src)

    def __getitem__(self, i):
        return {
            "src": torch.tensor(self.src[i], dtype=torch.long),
            "tgt": torch.tensor(self.tgt[i], dtype=torch.long),
        }


def collate_fn(batch):
    srcs = [b["src"] for b in batch]
    tgts = [b["tgt"] for b in batch]
    ms, mt = max(len(s) for s in srcs), max(len(t) for t in tgts)
    ps = torch.full((len(batch), ms), PAD_ID, dtype=torch.long)
    pt = torch.full((len(batch), mt), PAD_ID, dtype=torch.long)
    for i, s in enumerate(srcs):
        ps[i, :len(s)] = s
    for i, t in enumerate(tgts):
        pt[i, :len(t)] = t
    return {"src": ps, "tgt": pt}


train_loader = DataLoader(
    SummarizationDataset(train_df), batch_size=CONFIG["batch_size"],
    shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    SummarizationDataset(val_df), batch_size=CONFIG["batch_size"],
    shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True,
)


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (Vaswani et al., 2017)."""

    def __init__(self, d_model, max_len, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])


class TransformerSummarizer(nn.Module):
    """Encoder-decoder Transformer with weight-tied output projection."""

    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.d_model = cfg["d_model"]
        self.embedding = nn.Embedding(vocab_size, cfg["d_model"], padding_idx=PAD_ID)
        self.pos_encoder = PositionalEncoding(cfg["d_model"], cfg["max_pos"], cfg["dropout"])
        self.transformer = nn.Transformer(
            d_model=cfg["d_model"], nhead=cfg["nhead"],
            num_encoder_layers=cfg["num_enc"], num_decoder_layers=cfg["num_dec"],
            dim_feedforward=cfg["dim_ff"], dropout=cfg["dropout"],
            activation="gelu", batch_first=True, norm_first=False,
        )
        self.fc_out = nn.Linear(cfg["d_model"], vocab_size, bias=False)
        self.fc_out.weight = self.embedding.weight  # weight tying
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, tgt, tgt_mask=None, src_key_padding_mask=None):
        se = self.pos_encoder(self.embedding(src) * math.sqrt(self.d_model))
        te = self.pos_encoder(self.embedding(tgt) * math.sqrt(self.d_model))
        if tgt_mask is None:
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(
                tgt.size(1), device=src.device,
            )
        out = self.transformer(
            se, te, tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )
        return self.fc_out(out)


model = TransformerSummarizer(CONFIG, VOCAB_SIZE).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")


# ------------------------------------------------------------
# Optimizer, scheduler, loss, scaler
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=CONFIG["label_smooth"])
optimizer = torch.optim.AdamW(
    model.parameters(), lr=CONFIG["lr_peak"],
    betas=(0.9, 0.98), weight_decay=0.01,
)


def lr_lambda(step):
    """Linear warmup followed by inverse square-root decay."""
    step = max(step, 1)
    if step < CONFIG["warmup"]:
        return step / CONFIG["warmup"]
    return math.sqrt(CONFIG["warmup"] / step)


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=True)
except (TypeError, AttributeError):
    scaler = torch.cuda.amp.GradScaler(enabled=True)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------
@torch.no_grad()
def validate():
    """Return token-averaged validation loss."""
    model.eval()
    tot, ntok = 0.0, 0
    for batch in val_loader:
        src = batch["src"].to(device, non_blocking=True)
        tgt = batch["tgt"].to(device, non_blocking=True)
        ti, to = tgt[:, :-1], tgt[:, 1:]
        pad_mask = (src == PAD_ID)
        causal = nn.Transformer.generate_square_subsequent_mask(ti.size(1), device=device)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(src, ti, tgt_mask=causal, src_key_padding_mask=pad_mask)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), to.reshape(-1))
        nt = (to != PAD_ID).sum().item()
        tot += loss.item() * nt
        ntok += nt
    return tot / max(ntok, 1)


# ------------------------------------------------------------
# Training with early stopping
# ------------------------------------------------------------
best_val = float("inf")
no_improve = 0

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    running, ntok = 0.0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}", leave=False)

    for batch in pbar:
        src = batch["src"].to(device, non_blocking=True)
        tgt = batch["tgt"].to(device, non_blocking=True)
        ti, to = tgt[:, :-1], tgt[:, 1:]
        pad_mask = (src == PAD_ID)
        causal = nn.Transformer.generate_square_subsequent_mask(ti.size(1), device=device)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(src, ti, tgt_mask=causal, src_key_padding_mask=pad_mask)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), to.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        nt = (to != PAD_ID).sum().item()
        running += loss.item() * nt
        ntok += nt
        pbar.set_postfix(loss=f"{running / max(ntok, 1):.4f}")

    train_loss = running / max(ntok, 1)
    val_loss = validate()
    print(f"Epoch {epoch:02d} | train {train_loss:.4f} | val {val_loss:.4f}")

    if val_loss < best_val - 1e-4:
        best_val = val_loss
        no_improve = 0
        torch.save(
            {"model_state": model.state_dict(), "epoch": epoch, "val_loss": val_loss},
            CKPT_PATH,
        )
        print(f"  Saved best -> {CKPT_PATH}")
    else:
        no_improve += 1
        if no_improve >= CONFIG["patience"]:
            print("Early stopping.")
            break

print(f"Best validation loss: {best_val:.4f}")


# ------------------------------------------------------------
# Reload best checkpoint
# ------------------------------------------------------------
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=True)
model.load_state_dict(ckpt["model_state"])
model.to(device).eval()
print(f"Reloaded best (epoch {ckpt['epoch']}, val {ckpt['val_loss']:.4f})")


# ------------------------------------------------------------
# Beam-search inference
# ------------------------------------------------------------
@torch.no_grad()
def beam_generate(model, src_batch, cfg, device):
    """Beam search over the decoder for a batch of source sequences."""
    B = src_batch.size(0)
    num_beams = cfg["beam_size"]
    max_new = cfg["max_tgt_len"] - 1

    src_pad = (src_batch == PAD_ID)
    src_batch = src_batch.repeat_interleave(num_beams, dim=0)
    src_pad = src_pad.repeat_interleave(num_beams, dim=0)

    tgt = torch.full((B * num_beams, 1), BOS_ID, dtype=torch.long, device=device)
    scores = torch.full((B, num_beams), float("-inf"), device=device)
    scores[:, 0] = 0.0
    scores = scores.view(-1)
    finished = torch.zeros(B, dtype=torch.bool, device=device)

    for _ in range(max_new):
        L = tgt.size(1)
        causal = nn.Transformer.generate_square_subsequent_mask(L, device=device)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(src_batch, tgt, tgt_mask=causal, src_key_padding_mask=src_pad)
        logp = torch.log_softmax(logits[:, -1, :], dim=-1)

        cand = (scores.unsqueeze(1) + logp).view(B, num_beams * VOCAB_SIZE)
        topk_scores, topk_ids = cand.topk(num_beams, dim=-1)
        sel_scores = topk_scores.reshape(-1)
        sel_ids    = topk_ids.reshape(-1)
        beam_idx   = sel_ids // VOCAB_SIZE
        tok_idx    = sel_ids % VOCAB_SIZE

        tgt_3d = tgt.view(B, num_beams, -1)
        gather_idx = beam_idx.view(B, num_beams, 1).expand(-1, -1, tgt_3d.size(-1))
        tgt_3d = torch.gather(tgt_3d, 1, gather_idx)
        tgt = torch.cat([tgt_3d.reshape(B * num_beams, -1), tok_idx.unsqueeze(1)], dim=1)
        scores = sel_scores

        finished |= (tok_idx.view(B, num_beams) == EOS_ID).all(dim=1)
        if finished.all():
            break

    tgt = tgt.view(B, num_beams, -1)
    best = scores.view(B, num_beams).argmax(dim=-1)
    return tgt[torch.arange(B, device=device), best, :]


def decode_row(ids):
    return sp.decode([t for t in ids if t not in (PAD_ID, BOS_ID, EOS_ID)])


# ------------------------------------------------------------
# Batch inference over test set
# ------------------------------------------------------------
test_loader = DataLoader(
    SummarizationDataset(test_df), batch_size=CONFIG["eval_batch"],
    shuffle=False, collate_fn=collate_fn,
)
records = []
for batch in tqdm(test_loader, desc="Generating"):
    gen = beam_generate(model, batch["src"].to(device), CONFIG, device)
    for row in gen.cpu().tolist():
        records.append(decode_row(row))

pred_df = pd.DataFrame({
    "source":    test_df["content"].astype(str).tolist(),
    "reference": test_df["summary"].astype(str).tolist(),
    "predicted": records,
})
pred_df.to_csv(PRED_CSV, index=False, encoding="utf-8")
print(f"Saved {len(pred_df)} predictions -> {PRED_CSV}")


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------
pred_df = pd.read_csv(PRED_CSV).fillna("")
empty = pred_df["predicted"].astype(str).str.strip() == ""
n_empty = int(empty.sum())
if n_empty:
    pred_df.loc[empty, "predicted"] = "।"

refs = pred_df["reference"].astype(str).tolist()
hyps = pred_df["predicted"].astype(str).tolist()

# SacreBLEU
bleu_f  = BLEU(tokenize="flores200", max_ngram_order=4).corpus_score(hyps, [refs]).score
bleu_13 = BLEU(tokenize="13a",      max_ngram_order=4).corpus_score(hyps, [refs]).score


# ROUGE (manual, F1)
def _ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


def rouge_n_f1(hyp, ref, n):
    h_ng, r_ng = _ngrams(hyp, n), _ngrams(ref, n)
    if not h_ng or not r_ng:
        return 0.0
    overlap = sum((Counter(h_ng) & Counter(r_ng)).values())
    p, r = overlap / len(h_ng), overlap / len(r_ng)
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


def rouge_l_f1(hyp, ref):
    m, n = len(hyp), len(ref)
    if m == 0 or n == 0:
        return 0.0
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = (
                dp[i - 1][j - 1] + 1 if hyp[i - 1] == ref[j - 1]
                else max(dp[i - 1][j], dp[i][j - 1])
            )
    lcs = dp[m][n]
    p, r = lcs / m, lcs / n
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


pairs = [(h.split(), r.split()) for h, r in zip(hyps, refs)]
r1 = float(np.mean([rouge_n_f1(h, r, 1) for h, r in pairs]))
r2 = float(np.mean([rouge_n_f1(h, r, 2) for h, r in pairs]))
rL = float(np.mean([rouge_l_f1(h, r)    for h, r in pairs]))

# BERTScore
P, R, F1 = bert_score_fn(hyps, refs, lang="ne", verbose=False, device=device.type)


# Jaccard
def _jaccard(A, B):
    if not A and not B:
        return 1.0
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)


def jaccard_word(a, b):
    return _jaccard(set(str(a).split()), set(str(b).split()))


def jaccard_char_bigram(a, b):
    a, b = str(a), str(b)
    A = {a[i:i + 2] for i in range(max(len(a) - 1, 0))}
    B = {b[i:i + 2] for i in range(max(len(b) - 1, 0))}
    return _jaccard(A, B)


jac_w = float(np.mean([jaccard_word(h, r) for h, r in zip(hyps, refs)]))
jac_c = float(np.mean([jaccard_char_bigram(h, r) for h, r in zip(hyps, refs)]))


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
summary = {
    "model":               "Transformer Seq2Seq",
    "sacrebleu_flores200": round(bleu_f,  4),
    "sacrebleu_13a":       round(bleu_13, 4),
    "rouge1":              round(r1, 4),
    "rouge2":              round(r2, 4),
    "rougeL":              round(rL, 4),
    "bertscore_p":         round(P.mean().item(),  4),
    "bertscore_r":         round(R.mean().item(),  4),
    "bertscore_f1":        round(F1.mean().item(), 4),
    "jaccard_word":        round(jac_w, 4),
    "jaccard_char":        round(jac_c, 4),
    "n_samples":           len(refs),
    "n_empty":             n_empty,
    "best_val_loss":       round(best_val, 4),
}

print("=" * 55)
print("  Transformer Seq2Seq — FINAL METRICS")
print("=" * 55)
for k, v in summary.items():
    print(f"  {k:<22}: {v}")
print("=" * 55)
pd.Series(summary).to_csv(SCORES_CSV)
print(f"Saved -> {SCORES_CSV}")